In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import numpy as np

2026-05-06 19:00:53.352763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778094053.569559      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778094053.628087      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778094054.126277      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778094054.126320      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778094054.126322      22 computation_placer.cc:177] computation placer alr

In [2]:
train_dir = "/kaggle/input/datasets/mistblade69/brisc-dataset2025/brisc2025/classification_task/train"
test_dir = "/kaggle/input/datasets/mistblade69/brisc-dataset2025/brisc2025/classification_task/test"

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.15,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.15,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)



Found 5000 files belonging to 4 classes.
Using 4250 files for training.


I0000 00:00:1778094082.166794      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 5000 files belonging to 4 classes.
Using 750 files for validation.
Found 1000 files belonging to 4 classes.


In [4]:
print(class_names)

['glioma', 'meningioma', 'no_tumor', 'pituitary']


In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.20),
    layers.RandomZoom(0.2),
])

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [7]:
all_labels = np.concatenate(
    [y.numpy() for x, y in train_ds],
    axis=0
)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)

class_weights = dict(enumerate(class_weights))



In [8]:
print(class_weights)

{0: np.float64(1.0732323232323233), 1: np.float64(0.9328358208955224), 2: np.float64(1.210136674259681), 3: np.float64(0.8547868061142397)}


In [9]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [10]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = data_augmentation(inputs)

x = preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.5)(x)

x = layers.Dense(256, activation='relu')(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

In [11]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    )
]

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15


I0000 00:00:1778094105.916671      72 cuda_dnn.cc:529] Loaded cuDNN version 91002


133/133 ━━━━━━━━━━━━━━━━━━━━ 29s 130ms/step - accuracy: 0.4506 - loss: 1.6030 - val_accuracy: 0.7880 - val_loss: 0.5828 - learning_rate: 1.0000e-04
Epoch 2/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 104ms/step - accuracy: 0.6877 - loss: 0.8828 - val_accuracy: 0.8373 - val_loss: 0.4320 - learning_rate: 1.0000e-04
Epoch 3/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 104ms/step - accuracy: 0.7568 - loss: 0.6765 - val_accuracy: 0.8560 - val_loss: 0.3973 - learning_rate: 1.0000e-04
Epoch 4/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - accuracy: 0.7645 - loss: 0.6476 - val_accuracy: 0.8693 - val_loss: 0.3430 - learning_rate: 1.0000e-04
Epoch 5/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 103ms/step - accuracy: 0.7822 - loss: 0.6377 - val_accuracy: 0.8720 - val_loss: 0.3614 - learning_rate: 1.0000e-04
Epoch 6/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.7965 - loss: 0.5594
Epoch 6: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
133/133 ━━━━━━━━━━━━━━━━━━━━ 14s 103ms/step - accura

In [14]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [15]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 35s 156ms/step - accuracy: 0.7876 - loss: 0.6446 - val_accuracy: 0.8600 - val_loss: 0.5324 - learning_rate: 1.0000e-04
Epoch 2/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 132ms/step - accuracy: 0.8827 - loss: 0.3337 - val_accuracy: 0.9093 - val_loss: 0.3690 - learning_rate: 1.0000e-04
Epoch 3/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 133ms/step - accuracy: 0.9177 - loss: 0.2359 - val_accuracy: 0.9240 - val_loss: 0.3305 - learning_rate: 1.0000e-04
Epoch 4/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 132ms/step - accuracy: 0.9249 - loss: 0.2098 - val_accuracy: 0.9013 - val_loss: 0.3651 - learning_rate: 1.0000e-04
Epoch 5/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 132ms/step - accuracy: 0.9338 - loss: 0.1846 - val_accuracy: 0.9227 - val_loss: 0.2347 - learning_rate: 1.0000e-04
Epoch 6/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 133ms/step - accuracy: 0.9484 - loss: 0.1469 - val_accuracy: 0.9307 - val_loss: 0.2331 - learning_rate: 1.0000e-04
Epoch 7/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 18s 13

In [17]:
test_loss, test_acc = model.evaluate(test_ds)

32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - accuracy: 0.9565 - loss: 0.1560


In [18]:
print("Test Accuracy:", test_acc)

Test Accuracy: 0.9610000252723694


In [19]:
y_true = np.concatenate(
    [y.numpy() for x, y in test_ds],
    axis=0
)

y_pred_probs = model.predict(test_ds)

y_pred = np.argmax(y_pred_probs, axis=1)

32/32 ━━━━━━━━━━━━━━━━━━━━ 7s 144ms/step


In [20]:
print(confusion_matrix(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

[[244   7   0   3]
 [  9 278   1  18]
 [  0   0 140   0]
 [  0   1   0 299]]
              precision    recall  f1-score   support

      glioma       0.96      0.96      0.96       254
  meningioma       0.97      0.91      0.94       306
    no_tumor       0.99      1.00      1.00       140
   pituitary       0.93      1.00      0.96       300

    accuracy                           0.96      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.96      0.96      0.96      1000



In [21]:
model.save("/kaggle/working/resnet50_tumor_model.keras")

print("Model Saved")

Model Saved


In [22]:
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

# class names
class_names_check = ['glioma', 'meningioma', 'pituitary', 'no_tumor']

# load model
model_check = load_model("resnet50_tumor_model.keras")

# load image
img_path_check = "/kaggle/input/datasets/mistblade69/brisc-dataset2025/brisc2025/classification_task/test/glioma/brisc2025_test_00001_gl_ax_t1.jpg"

img_check = load_img(img_path_check, target_size=(224,224))

# convert to array
img_array_check = img_to_array(img_check)

# add batch dimension
img_array_check = np.expand_dims(img_array_check, axis=0)

# preprocess for ResNet50
img_array_check = preprocess_input(img_array_check)

# prediction
predictions_check = model.predict(img_array_check)

# predicted class
predicted_class_check = np.argmax(predictions_check)

print("Predicted Class:", class_names_check[predicted_class_check])

print("Confidence:", np.max(predictions_check))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Class: glioma
Confidence: 0.9999925
